Task 1
Imagine you are a data scientist at a real estate company. Predict house prices using square footage, bedrooms, bathrooms, age, and neighborhood. Clean the dataset, encode categorical values, identify relevant features, evaluate model performance, and predict the price for a new home.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, export_text

np.random.seed(42)
locations = ['Downtown', 'Suburb', 'Riverside']
data = pd.DataFrame({
    'sqft': np.random.randint(600, 4000, 150),
    'bedrooms': np.random.randint(1, 6, 150),
    'bathrooms': np.random.randint(1, 4, 150),
    'age': np.random.randint(0, 60, 150),
    'neighborhood': np.random.choice(locations, 150),
})
noise = np.random.normal(0, 25000, 150)
location_values = {'Downtown': 50000, 'Suburb': 20000, 'Riverside': 30000}
data['price'] = 50000 + data['sqft'] * 120 + data['bedrooms'] * 15000 + data['bathrooms'] * 10000 - data['age'] * 800 + data['neighborhood'].map(location_values) + noise
indices = np.random.choice(data.index, 8, replace=False)
data.loc[indices[:3], 'sqft'] = np.nan
data.loc[indices[3:5], 'bathrooms'] = np.nan
data.loc[indices[5:], 'neighborhood'] = np.nan
data['sqft'] = data['sqft'].fillna(data['sqft'].median())
data['bathrooms'] = data['bathrooms'].fillna(data['bathrooms'].median())
data['neighborhood'] = data['neighborhood'].fillna(data['neighborhood'].mode()[0])
encoded = pd.get_dummies(data['neighborhood'], prefix='neighborhood')
data = pd.concat([data.drop(columns=['neighborhood']), encoded], axis=1)
correlations = data.corr()['price'].abs().sort_values(ascending=False)
print('Task 1: House price prediction')
print('Top relevant features:')
print(correlations.head(6))
features = ['sqft', 'bedrooms', 'bathrooms', 'age', 'neighborhood_Downtown', 'neighborhood_Riverside', 'neighborhood_Suburb']
X = data[features]
y = data['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print('RMSE:', np.sqrt(mean_squared_error(y_test, y_pred)))
print('R2:', r2_score(y_test, y_pred))
new_house = pd.DataFrame([{
    'sqft': 1800,
    'bedrooms': 3,
    'bathrooms': 2,
    'age': 10,
    'neighborhood_Downtown': 0,
    'neighborhood_Riverside': 1,
    'neighborhood_Suburb': 0,
}])
print('Predicted price for new house:', model.predict(new_house)[0])

Task 2
You work for an email service provider and need to classify emails as spam or not spam based on content and sender information. Preprocess the dataset, train a classifier, evaluate performance, and classify a new incoming email.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, export_text

emails = [
    'Win money now',
    'Meeting schedule for tomorrow',
    'Your account has been compromised',
    'Lowest price for meds',
    'Project update and report attached',
    'Earn cash fast with this secret',
    'Family dinner plans tonight',
    'Limited time offer just for you',
    'Invoice for your purchase',
    'Claim your free gift card',
]
labels = [1, 0, 1, 1, 0, 1, 0, 1, 0, 1]
links = [1, 0, 1, 0, 0, 1, 0, 1, 0, 1]
lengths = [15, 29, 28, 21, 32, 30, 27, 30, 31, 28]
senders = ['unknown', 'trusted', 'unknown', 'marketing', 'trusted', 'spam', 'friend', 'promo', 'store', 'spam']
vectorizer = CountVectorizer()
text_features = vectorizer.fit_transform(emails).toarray()
text_df = pd.DataFrame(text_features, columns=vectorizer.get_feature_names_out())
email_df = pd.DataFrame({
    'hyperlink': links,
    'length': lengths,
    'sender': senders,
    'label': labels,
})
sender_map = {cat: code for code, cat in enumerate(pd.Series(senders, dtype='category').cat.categories)}
email_df = pd.concat([email_df, text_df], axis=1)
email_df['sender'] = email_df['sender'].map(sender_map)
X = email_df.drop(columns=['label'])
y = email_df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
spam_clf = LogisticRegression(max_iter=1000)
spam_clf.fit(X_train, y_train)
y_pred = spam_clf.predict(X_test)
print('Task 2: Email spam classification')
print('Accuracy:', accuracy_score(y_test, y_pred))
print('Precision:', precision_score(y_test, y_pred))
print('Recall:', recall_score(y_test, y_pred))
print('F1 score:', f1_score(y_test, y_pred))
new_email = ['free cash offer now']
new_text = vectorizer.transform(new_email).toarray()
new_sender = sender_map['unknown']
new_x = np.hstack([np.array([[1, 23, new_sender]]), new_text])
print('New email prediction (1=spam,0=not spam):', spam_clf.predict(new_x))

Task 3
You work for a retail business and must classify customers as high-value or low-value based on spending, age, visits, and purchase frequency. Clean the data, handle missing values and outliers, split into training and testing sets, find a separating hyperplane, generate classification rules, and evaluate performance.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, export_text

np.random.seed(42)

customer_data = pd.DataFrame({
    'total_spending': np.random.normal(1000, 300, 200).clip(100, 2500),
    'age': np.random.randint(18, 70, 200),
    'visits': np.random.poisson(8, 200).clip(1, 30),
    'frequency': np.random.uniform(0.1, 1.0, 200),
})
customer_data.loc[np.random.choice(customer_data.index, 10), 'total_spending'] = np.nan
customer_data.loc[np.random.choice(customer_data.index, 10), 'visits'] = 999
customer_data['label'] = ((customer_data['total_spending'].fillna(1000) * 0.5 + customer_data['visits'] * 30 + customer_data['frequency'] * 200) > 900).astype(int)
customer_data['total_spending'] = customer_data['total_spending'].fillna(customer_data['total_spending'].median())
customer_data['visits'] = customer_data['visits'].replace(999, customer_data['visits'].median())
customer_data['age'] = customer_data['age'].clip(18, 70)
X = customer_data[['total_spending', 'age', 'visits', 'frequency']]
y = customer_data['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
clf = SVC(kernel='linear', probability=True, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
rules = DecisionTreeClassifier(max_depth=3, random_state=42)
rules.fit(X_train, y_train)
print('Task 3: Customer value classification')
print('Accuracy:', accuracy_score(y_test, y_pred))
print('Precision:', precision_score(y_test, y_pred))
print('Recall:', recall_score(y_test, y_pred))
print('F1 score:', f1_score(y_test, y_pred))
print('Decision tree rules:')
print(export_text(rules, feature_names=list(X.columns)))